Đây là script chạy phân tích và đánh giá mô hình Chrono-G gồm các bước:
- Clone repository từ GitHub
- Load dữ liệu đề bài của cuộc thi và trọng số mô hình
- Chạy phân tích mô hình

**Lưu ý**:
- Script này không thể chạy local do ngốn bộ nhớ. Script **không bắt buộc** phải chạy trên GPU, chúng tôi có đính kèm hướng dẫn chi tiết chạy trên [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Notebook chạy cho mô hình Chrono-G của dự án ChronoLens. Việc thay đổi sang mô hình khác có thể dẫn tới việc notebook không chạy được do sự khác biệt về kiến trúc mô hình.
- Thời gian chạy cụ thể: 20-30 phút
- Với những trường hợp có thời gian chạy quá lớn (>30 phút), chúng tôi khuyến khích nên sử dụng cơ chế chạy offline trên Kaggle.
- Cell code 9 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)

In [ ]:
!git clone https://github.com/CryAndRRich/chronolens.git

In [ ]:
%cd /kaggle/working/chronolens
CHRONO_PATH = "/kaggle/working/chronolens"

In [ ]:
!pip install -r requirements.txt

In [4]:
import sys

if CHRONO_PATH not in sys.path:
    sys.path.append(CHRONO_PATH)

In [5]:
import torch

In [6]:
from config import *
from preprocess import *
from model import *
from explainer import *
from utils import *

In [ ]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [8]:
INPUT_ROOT = "YOUR_INTPUT_PATH"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

MODEL_NAME = "chrono_g"
CHECKPOINT_FILE = f"{INPUT_ROOT}/weights/best_model_{MODEL_NAME}.pth"

In [9]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [10]:
train_loader, val_loader, test_loader = data_manager.get_dataloaders()

In [ ]:
model_kwargs = update_model_kwargs(
    data_manager=data_manager,
    model_kwargs=CONFIG_MODEL.MODEL_KWARGS[MODEL_NAME]
)
model = get_model(
    name=MODEL_NAME,
    **model_kwargs  
).to(DEVICE)

model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))

In [12]:
top_edges = extract_graph_edges(
    model, 
    val_loader, 
    id_to_idx=data_manager.id_to_idx, 
    layer_idx=0,
    device=DEVICE, 
)

In [ ]:
plot_graph_network(top_edges, top_k=50, max_occurrences=10)
plot_graph_network(top_edges, top_k=100, max_occurrences=10)
plot_graph_network(top_edges, top_k=150, max_occurrences=10)
plot_graph_network(top_edges, top_k=200, max_occurrences=10)

In [ ]:
correct_attn, wrong_attn = extract_error_attention(
    model, 
    val_loader, 
    target_task_idx=1, 
    id_to_idx=data_manager.id_to_idx, 
    device=DEVICE
)

distractors = []
for action_id in wrong_attn:
    if action_id in correct_attn:
        diff = wrong_attn[action_id] - correct_attn[action_id]
        if diff > 0:
            distractors.append((action_id, diff, wrong_attn[action_id], correct_attn[action_id]))

if len(distractors) > 0:
    distractors.sort(key=lambda x: x[1], reverse=True)
    
    for i, (action_id, diff, wrong_w, correct_w) in enumerate(distractors[:3]):
        print(f"Top {i + 1}: Mã {action_id}")
        print(f"   - Khi đoán Sai, model tập trung vào nó: {wrong_w:.2f}%")
        print(f"   - Khi đoán Đúng, model chỉ tập trung:   {correct_w:.2f}%")
        print(f"   => Kết luận: Hành vi gây nhiễu, dễ khiến mô hình bị 'ảo tưởng'.")
    
    plot_distractor_analysis(distractors, top_k=5)